# 08 — Ablation Study

Removes one component at a time from M8 (the full system) and measures the AP_hard drop.

**Ablation variants:**

| Variant | Component removed | Expected category |
|---|---|---|
| M8_no_fem | FEM module | arch |
| M8_no_popam | POPAM module | arch |
| M8_no_depth | CrossAttn depth fusion | depth |
| M8_no_aug | Label-aware augmentation | aug |
| M8_no_vishead | VisibilityMapHead | arch |

Each variant trains from scratch with identical hyperparameters except the removed component.

**Output:**
- Horizontal ablation bar chart (`results/figures/ablation_bar_chart.{png,pdf}`)
- Narrative template for each ablation variant

**Prerequisite:** notebook `07_full_system` must have completed (M8 trained).

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import load_config, set_all_seeds
from src.datasets import KITTIDataset, collate_fn
from src.logger import ExperimentLogger
from src.metrics import compute_kitti_ap, sample_to_annotation
from src.plotting import plot_ablation_bar_chart, create_narrative_template

ROOT = Path('..').resolve()
TABLES_DIR = ROOT / 'results' / 'tables'
FIGURES_DIR = ROOT / 'results' / 'figures'
NARRATIVES_DIR = ROOT / 'results' / 'narratives'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
NARRATIVES_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = 'local'   # change to 'kaggle' for full training
SMOKE_TEST = (RUN_MODE == 'local')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load M8 AP_hard from results table
results_csv = TABLES_DIR / 'val_results_all_models.csv'
if results_csv.exists():
    df_all = pd.read_csv(results_csv)
    m8_row = df_all[df_all['model_id'] == 'M8']
    M8_AP_HARD = float(m8_row['AP_hard'].values[0]) if len(m8_row) > 0 else None
    print(f'M8 AP_hard (reference): {M8_AP_HARD}')
else:
    M8_AP_HARD = None
    print('[WARN] val_results_all_models.csv not found. Run notebooks 01–07 first.')

print(f'Device: {device}  |  Mode: {RUN_MODE}')

In [ ]:
def run_ablation(variant_id, overrides, description, category):
    """Train one ablation variant and return (ap_hard, delta_ap_hard)."""
    from ultralytics import YOLO

    base_overrides = {
        'run_mode': RUN_MODE,
        'model_id': variant_id,
        'epochs': 3 if SMOKE_TEST else 100,
        'data_limit': 100 if SMOKE_TEST else None,
    }
    base_overrides.update(overrides)

    cfg = load_config(ROOT / 'configs' / 'full_system.yaml', overrides=base_overrides)
    cfg.ensure_dirs()
    set_all_seeds(cfg.seed)

    val_ds = KITTIDataset(cfg.kitti_root, split='val', imgsz=cfg.imgsz)
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch, shuffle=False,
        collate_fn=collate_fn, num_workers=0
    )

    model = YOLO('yolov8s.pt')
    logger = ExperimentLogger(
        run_id=f'{variant_id}_seed{cfg.seed}',
        log_dir=cfg.log_dir,
        config=vars(cfg),
    )

    with logger:
        model.train(
            data=str(ROOT / 'data' / 'kitti_ultralytics.yaml'),
            epochs=cfg.epochs,
            imgsz=cfg.imgsz,
            batch=cfg.batch,
            optimizer=cfg.optimizer,
            lr0=cfg.lr0,
            lrf=cfg.lrf,
            momentum=cfg.momentum,
            weight_decay=cfg.weight_decay,
            warmup_epochs=cfg.warmup_epochs,
            amp=cfg.amp,
            seed=cfg.seed,
            project=str(cfg.checkpoint_dir),
            name=variant_id,
            exist_ok=True,
            device=device,
        )

        best_ckpt = cfg.checkpoint_dir / variant_id / 'weights' / 'best.pt'
        best_model = YOLO(str(best_ckpt))
        best_model.model.eval()

        predictions, annotations = [], []
        with torch.no_grad():
            for batch in val_loader:
                imgs = batch['image'].to(device)
                results = best_model(imgs, verbose=False)
                for i, r in enumerate(results):
                    boxes = r.boxes.xyxyn.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
                    scores = r.boxes.conf.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
                    predictions.append({'image_id': batch['image_id'][i], 'boxes': boxes, 'scores': scores})
                    annotations.append(sample_to_annotation(batch, i))

        ap_easy = compute_kitti_ap(predictions, annotations, 'easy')
        ap_mod  = compute_kitti_ap(predictions, annotations, 'moderate')
        ap_hard = compute_kitti_ap(predictions, annotations, 'hard')
        logger.log_metrics({'AP_easy': ap_easy, 'AP_mod': ap_mod, 'AP_hard': ap_hard})

    delta = (ap_hard - M8_AP_HARD) if M8_AP_HARD is not None else None
    create_narrative_template(variant_id, NARRATIVES_DIR)
    print(f'{variant_id}: AP_hard={ap_hard:.2f}  delta={delta:+.2f}' if delta is not None else f'{variant_id}: AP_hard={ap_hard:.2f}')
    return ap_easy, ap_mod, ap_hard, delta, description, category

In [ ]:
# ── Ablation variants ─────────────────────────────────────────────────────────
# Define: (variant_id, overrides_dict, description, category)
ABLATION_VARIANTS = [
    ('M8_no_fem',     {'use_fem': False},                            '-FEM',            'arch'),
    ('M8_no_popam',   {'use_popam': False},                          '-POPAM',          'arch'),
    ('M8_no_depth',   {'fusion_type': 'none'},                       '-CrossAttnDepth', 'depth'),
    ('M8_no_aug',     {'aug_probability': 0.0, 'label_aware': False},'-LabelAug',       'aug'),
    ('M8_no_vishead', {'use_visibility_head': False,
                       'lambda_vis': 0.0,
                       'lambda_occ_consistency': 0.0},              '-VisHead',        'arch'),
]

ablation_results = []
for variant_id, overrides, description, category in ABLATION_VARIANTS:
    print(f'\n{"="*60}\nAblation: {variant_id}\n{"="*60}')
    ap_easy, ap_mod, ap_hard, delta, desc, cat = run_ablation(
        variant_id, overrides, description, category
    )
    ablation_results.append({
        'variant_id': variant_id,
        'description': desc,
        'category': cat,
        'AP_easy': ap_easy,
        'AP_mod': ap_mod,
        'AP_hard': ap_hard,
        'delta_AP_hard': delta,
    })

abl_df = pd.DataFrame(ablation_results)
abl_df = abl_df.sort_values('delta_AP_hard')  # most negative drop first
print('\nAblation results:')
print(abl_df[['variant_id', 'AP_hard', 'delta_AP_hard', 'category']].to_string(index=False))

In [ ]:
# ── Plot ablation bar chart ────────────────────────────────────────────────────
# Colour-coded by category: depth=teal, aug=orange, arch=blue
# Sorted by drop magnitude (largest drop first)

plot_ablation_bar_chart(
    ablation_data=abl_df.to_dict('records'),
    m8_ap_hard=M8_AP_HARD,
    save_dir=FIGURES_DIR,
)
print(f'Saved: {FIGURES_DIR / "ablation_bar_chart.png"}')

In [ ]:
# ── Save ablation results ──────────────────────────────────────────────────────
abl_csv = TABLES_DIR / 'ablation_results.csv'
abl_df.to_csv(abl_csv, index=False)
print(f'Saved: {abl_csv}')

# Also append to val_results_all_models.csv for completeness
if results_csv.exists():
    df_main = pd.read_csv(results_csv)
    abl_rows = abl_df.rename(columns={'variant_id': 'model_id'}).copy()
    abl_rows['aug_p'] = None
    abl_rows['use_fem'] = None
    abl_rows['use_popam'] = None
    abl_rows['fusion_type'] = None
    abl_rows['use_vis_head'] = None
    abl_rows['seed'] = 42
    abl_rows['epochs'] = 100 if not SMOKE_TEST else 3
    df_main = df_main[~df_main['model_id'].isin(abl_df['variant_id'].tolist())]
    df_main = pd.concat([df_main, abl_rows[df_main.columns.tolist()]], ignore_index=True)
    df_main.to_csv(results_csv, index=False)
    print(f'Ablation rows appended to: {results_csv}')